# NCTB-SchoolText Audit Report

Aggregates chapter/chunk/flagged-chunk statistics from the Gemma-based OCR audit results,
across all classes and subjects. Subjects are marked **Salvageable** if their flagged-chunk
percentage is at or below the configured threshold, otherwise **Non-salvageable**.


In [1]:
import json
import re
from pathlib import Path
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.width', 160)


## Configuration

`BASE_DIR` is the root folder containing all `classX` folders.
`CLASS_SUBJECTS` maps each class folder name to the list of its subject folders.
`SALVAGEABLE_THRESHOLD` is the max allowed flagged-chunk percentage for a subject
to be considered salvageable.


In [2]:
BASE_DIR = Path(r"D:\Ongoing Research Works\AI-Tutor\NCTB-SchoolText")

SALVAGEABLE_THRESHOLD = 30.0  # percent

CLASS_SUBJECTS = {
    "classOne": [
        "processed_chapters_bangla",
        "processed_chapters_english",
        "processed_chapters_math",
    ],
    "classTwo": [
        "processed_chapters_bangla",
        "processed_chapters_english",
        "processed_chapters_math",
    ],
    "classThree": [
        "processed_chapters_bangla",
        "processed_chapters_bgs",
        "processed_chapters_buddhist_religion",
        "processed_chapters_christian_religion",
        "processed_chapters_english",
        "processed_chapters_hindu_religion",
        "processed_chapters_islam",
        "processed_chapters_math",
        "processed_chapters_science",
    ],
    "classFour": [
        "processed_chapters_bangla",
        "processed_chapters_bgs",
        "processed_chapters_buddhist_religion",
        "processed_chapters_christian_religion",
        "processed_chapters_english",
        "processed_chapters_hindu_religion",
        "processed_chapters_islam",
        "processed_chapters_math",
        "processed_chapters_science",
    ],
    "classFive": [
        "processed_chapters_bangla",
        "processed_chapters_bgs",
        "processed_chapters_buddhist_religion",
        "processed_chapters_christian_religion",
        "processed_chapters_eng",
        "processed_chapters_hindu_religion",
        "processed_chapters_islam",
        "processed_chapters_math",
        "processed_chapters_science",
    ],
    "classSix": [
        "processed_chapters_agriculture",
        "processed_chapters_arabic",
        "processed_chapters_arts_and_crafts",
        "processed_chapters_bangla",
        "processed_chapters_bangla_grammar",
        "processed_chapters_bangla_rapidreader",
        "processed_chapters_bgs",
        "processed_chapters_buddhist_religion",
        "processed_chapters_christian_religion",
        "processed_chapters_english",
        "processed_chapters_english_grammar",
        "processed_chapters_hindu_religion",
        "processed_chapters_home_science",
        "processed_chapters_ict",
        "processed_chapters_islam",
        "processed_chapters_math",
        "processed_chapters_music",
        "processed_chapters_pali",
        "processed_chapters_physical_education",
        "processed_chapters_sanskrit",
        "processed_chapters_science",
        "processed_chapters_work_and_life",
    ],
    "classSeven": [
        "processed_chapters_agriculture",
        "processed_chapters_arabic",
        "processed_chapters_arts_and_crafts",
        "processed_chapters_bangla",
        "processed_chapters_bangla_grammar",
        "processed_chapters_bangla_rapidreader",
        "processed_chapters_bgs",
        "processed_chapters_buddhist_religion",
        "processed_chapters_christian_religion",
        "processed_chapters_english",
        "processed_chapters_english_grammar",
        "processed_chapters_hindu_religion",
        "processed_chapters_home_science",
        "processed_chapters_ict",
        "processed_chapters_islam",
        "processed_chapters_math",
        "processed_chapters_music",
        "processed_chapters_pali",
        "processed_chapters_physical_education",
        "processed_chapters_sanskrit",
        "processed_chapters_science",
        "processed_chapters_work_and_life",
    ],
    "classEight": [
        "processed_chapters_agriculture",
        "processed_chapters_arabic",
        "processed_chapters_arts_and_crafts",
        "processed_chapters_bangla",
        "processed_chapters_bangla_grammar",
        "processed_chapters_bangla_rapidreader",
        "processed_chapters_bgs",
        "processed_chapters_buddhist_religion",
        "processed_chapters_christian_religion",
        "processed_chapters_english",
        "processed_chapters_english_grammar",
        "processed_chapters_hindu_religion",
        "processed_chapters_home_science",
        "processed_chapters_ict",
        "processed_chapters_islam",
        "processed_chapters_math",
        "processed_chapters_music",
        "processed_chapters_pali",
        "processed_chapters_physical_education",
        "processed_chapters_sanskrit",
        "processed_chapters_science",
        "processed_chapters_work_and_life",
    ],
    "classNineTen": [
        "processed_chapters_accounting",
        "processed_chapters_agriculture",
        "processed_chapters_arabic",
        "processed_chapters_arts_and_crafts",
        "processed_chapters_bangla",
        "processed_chapters_bangla_grammar",
        "processed_chapters_bangla_rapidreader",
        "processed_chapters_bgs",
        "processed_chapters_biology_secondary",
        "processed_chapters_buddhist_religion",
        "processed_chapters_business_entrepreneurship",
        "processed_chapters_career_education",
        "processed_chapters_chemistry_secondary",
        "processed_chapters_christian_religion",
        "processed_chapters_civics",
        "processed_chapters_economics",
        "processed_chapters_english",
        "processed_chapters_english_grammar",
        "processed_chapters_finance",
        "processed_chapters_geography",
        "processed_chapters_higher_math",
        "processed_chapters_hindu_religion",
        "processed_chapters_history",
        "processed_chapters_home_science",
        "processed_chapters_ict",
        "processed_chapters_islam",
        "processed_chapters_math",
        "processed_chapters_music",
        "processed_chapters_pali",
        "processed_chapters_physical_education",
        "processed_chapters_physics_secondary",
        "processed_chapters_sanskrit",
        "processed_chapters_science",
    ],
}


## Helper functions

- `chapters_in_config`: total chapters a subject *should* have, read from its
  `chapters_config_<subject>.json` file — this is the ground-truth curriculum total
  and doesn't depend on how much auditing is done yet.
- `chapters_audited` / `chapters_parse_error`: how many chapter audit files in
  `Audit Results` parsed successfully as a list of chunk results vs. failed to parse
  (e.g. saved raw text or an error dict from a failed API call).
- `chunks_total` / `chunks_flagged`: summed only from successfully parsed audit files.


In [3]:
def get_subject_name(subject_folder: str) -> str:
    return subject_folder.replace("processed_chapters_", "", 1)


def get_config_chapter_count(subject_dir: Path, subject_name: str):
    config_path = subject_dir / f"chapters_config_{subject_name}.json"
    if not config_path.exists():
        return None
    try:
        data = json.loads(config_path.read_text(encoding="utf-8"))
        if isinstance(data, list):
            return len(data)
    except json.JSONDecodeError:
        pass
    return None


def analyze_subject(class_name: str, subject_folder: str, subject_dir: Path):
    subject_name = get_subject_name(subject_folder)
    chapters_in_config = get_config_chapter_count(subject_dir, subject_name)

    audit_dir = subject_dir / "Audit Results"
    chapters_audited = 0
    chapters_parse_error = 0
    chunks_total = 0
    chunks_flagged = 0

    if audit_dir.exists():
        for audit_file in sorted(audit_dir.glob("*.json")):
            try:
                content = json.loads(audit_file.read_text(encoding="utf-8"))
            except json.JSONDecodeError:
                chapters_parse_error += 1
                continue

            if not isinstance(content, list):
                # e.g. an error dict saved from a failed request, or raw non-JSON text
                chapters_parse_error += 1
                continue

            chapters_audited += 1
            chunks_total += len(content)
            chunks_flagged += sum(
                1 for entry in content
                if isinstance(entry, dict) and entry.get("status") == "flagged"
            )

    pct_flagged = round((chunks_flagged / chunks_total * 100), 2) if chunks_total > 0 else None

    if chunks_total == 0:
        salvageable = "N/A (no audited chunks)"
    elif pct_flagged <= SALVAGEABLE_THRESHOLD:
        salvageable = "Salvageable"
    else:
        salvageable = "Non-salvageable"

    return {
        "class": class_name,
        "subject_folder": subject_folder,
        "subject": subject_name,
        "chapters_in_config": chapters_in_config,
        "chapters_audited": chapters_audited,
        "chapters_parse_error": chapters_parse_error,
        "chunks_total": chunks_total,
        "chunks_flagged": chunks_flagged,
        "pct_flagged": pct_flagged,
        "salvageable": salvageable,
    }


## Run the analysis

Walks every class/subject pair defined above and builds one row per subject.


In [4]:
rows = []
for class_name, subject_folders in CLASS_SUBJECTS.items():
    class_dir = BASE_DIR / class_name
    for subject_folder in subject_folders:
        subject_dir = class_dir / subject_folder
        if not subject_dir.exists():
            print(f"! Missing folder, skipping: {subject_dir}")
            continue
        rows.append(analyze_subject(class_name, subject_folder, subject_dir))

subject_df = pd.DataFrame(rows)
subject_df


,class,subject_folder,subject,chapters_in_config,chapters_audited,chapters_parse_error,chunks_total,chunks_flagged,pct_flagged,salvageable
0,classOne,processed_chapters_bangla,bangla,54.0,52,2,56,28,50.00,Non-salvageable
1,classOne,processed_chapters_english,english,5.0,5,0,49,41,83.67,Non-salvageable
2,classOne,processed_chapters_math,math,18.0,18,0,57,53,92.98,Non-salvageable
3,classTwo,processed_chapters_bangla,bangla,30.0,30,0,65,49,75.38,Non-salvageable
4,classTwo,processed_chapters_english,english,10.0,8,2,104,93,89.42,Non-salvageable
5,classTwo,processed_chapters_math,math,7.0,7,0,108,78,72.22,Non-salvageable
6,classThree,processed_chapters_bangla,bangla,30.0,30,0,153,70,45.75,Non-salvageable
7,classThree,processed_chapters_bgs,bgs,13.0,13,0,154,84,54.55,Non-salvageable
8,classThree,processed_chapters_buddhist_religion,buddhist_religion,9.0,9,0,184,90,48.91,Non-salvageable
9,classThree,processed_chapters_christian_religion,christian_religion,5.0,5,0,237,109,45.99,Non-salvageable


## 9–12. Per-subject, per-class stats (chapters, chunks, flagged, %, salvageable)


In [5]:
subject_report = subject_df[[
    "class", "subject", "chapters_in_config", "chapters_audited",
    "chunks_total", "chunks_flagged", "pct_flagged", "salvageable"
]].sort_values(["class", "subject"]).reset_index(drop=True)

subject_report


,class,subject,chapters_in_config,chapters_audited,chunks_total,chunks_flagged,pct_flagged,salvageable
0,classEight,agriculture,6.0,6,476,53,11.13,Salvageable
1,classEight,arabic,12.0,12,113,103,91.15,Non-salvageable
2,classEight,arts_and_crafts,8.0,8,158,14,8.86,Salvageable
3,classEight,bangla,17.0,0,0,0,NaN,N/A (no audited chunks)
4,classEight,bangla_grammar,15.0,15,483,195,40.37,Non-salvageable
5,classEight,bangla_rapidreader,15.0,15,255,40,15.69,Salvageable
6,classEight,bgs,13.0,12,572,112,19.58,Salvageable
7,classEight,buddhist_religion,11.0,10,430,74,17.21,Salvageable
8,classEight,christian_religion,10.0,10,421,43,10.21,Salvageable
9,classEight,english,11.0,10,510,101,19.80,Salvageable


## 5–8. Per-class stats (chapters, chunks, flagged, %)


In [6]:
class_df = subject_df.groupby("class", as_index=False).agg(
    chapters_in_config=("chapters_in_config", "sum"),
    chapters_audited=("chapters_audited", "sum"),
    chunks_total=("chunks_total", "sum"),
    chunks_flagged=("chunks_flagged", "sum"),
)

class_df["pct_flagged"] = class_df.apply(
    lambda r: round(r["chunks_flagged"] / r["chunks_total"] * 100, 2) if r["chunks_total"] > 0 else None,
    axis=1,
)

# preserve the class order as defined in CLASS_SUBJECTS rather than alphabetical
class_order = list(CLASS_SUBJECTS.keys())
class_df["class"] = pd.Categorical(class_df["class"], categories=class_order, ordered=True)
class_df = class_df.sort_values("class").reset_index(drop=True)

class_df


,class,chapters_in_config,chapters_audited,chunks_total,chunks_flagged,pct_flagged
0,classOne,77.0,75,162,122,75.31
1,classTwo,47.0,45,277,220,79.42
2,classThree,100.0,97,1629,860,52.79
3,classFour,99.0,87,1871,846,45.22
4,classFive,104.0,97,2290,1178,51.44
5,classSix,215.0,207,6401,1585,24.76
6,classSeven,217.0,211,7910,2248,28.42
7,classEight,215.0,196,7905,2065,26.12
8,classNineTen,450.0,414,22397,6839,30.54


## 1–4. Grand totals across all classes


In [7]:
total_chapters_config = int(subject_df["chapters_in_config"].sum(skipna=True))
total_chapters_audited = int(subject_df["chapters_audited"].sum())
total_chunks = int(subject_df["chunks_total"].sum())
total_flagged = int(subject_df["chunks_flagged"].sum())
overall_pct_flagged = round((total_flagged / total_chunks * 100), 2) if total_chunks > 0 else None

print(f"Total chapters (per chapter configs):      {total_chapters_config}")
print(f"Total chapters actually audited so far:    {total_chapters_audited}")
print(f"Total chunks audited:                      {total_chunks}")
print(f"Total flagged chunks:                      {total_flagged}")
print(f"Overall flagged percentage:                {overall_pct_flagged}%")


Total chapters (per chapter configs):      1524
Total chapters actually audited so far:    1429
Total chunks audited:                      50842
Total flagged chunks:                      15963
Overall flagged percentage:                31.4%


## Save reports to CSV (optional)


In [8]:
subject_report.to_csv(BASE_DIR / "audit_summary_by_subject.csv", index=False)
class_df.to_csv(BASE_DIR / "audit_summary_by_class.csv", index=False)
print("Saved CSV summaries to", BASE_DIR)


Saved CSV summaries to D:\Ongoing Research Works\AI-Tutor\NCTB-SchoolText
